In [1]:
import pandas as pd 
import os
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import json
import mne
import numpy as np
from tqdm import tqdm
import re
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from multiprocessing import Pool
import warnings
from collections import defaultdict
from typing import Dict, Tuple, Any, List

warnings.filterwarnings("ignore")

os.chdir('../../..')

Functions

In [2]:
    def _get_day_time_df(day_time_meta_path):
        # Читаем первый лист
        meta1 = pd.read_excel(day_time_meta_path, sheet_name="1_Сессия", header=1).rename(columns={
            'Subject ID'          : 'Subject_id',
            'Время начала записи' : 'Time',
        })
        meta1['Trial_id'] = 1
        
        # Читаем второй лист
        meta2 = pd.read_excel(day_time_meta_path, sheet_name="2_Сессия", header=1).rename(columns={
            'Subject ID'          : 'Subject_id',
            'Время начала записи' : 'Time',
        })
        meta2['Trial_id'] = 2
        
        # Конкатенация
        meta = pd.concat([meta1, meta2], ignore_index=True)
        
        meta
        
        meta['Subject_id'] = (
            meta['Subject_id']
            .astype(str)
            .str.extract(r'(\d+)', expand=False)
            .astype('float')
        )
        meta = meta[meta['Subject_id'].notna()].copy()
        meta['Subject_id'] = meta['Subject_id'].astype(int)
        
        s  = meta['Time'].astype(str).str.strip()                    
        n  = pd.to_numeric(s, errors='coerce')                        
        dt_str = pd.to_datetime(s, errors='coerce', infer_datetime_format=True)
        dt_num = pd.to_datetime(n, errors='coerce', origin='1899-12-30', unit='D')
        dt = dt_str.fillna(dt_num)                                   
        
        meta['Hour'] = dt.dt.hour
        meta['Time'] = dt.dt.strftime('%H:%M')
        
        meta_idx = (meta[['Subject_id', 'Trial_id', 'Hour', 'Time']]
                    .dropna(subset=['Hour'])                         
                    .drop_duplicates(['Subject_id', 'Trial_id'], keep='last'))
        
        meta_idx['Condition'] = pd.cut(
            meta_idx['Hour'],
            bins=[-0.1, 10, 18, 24],
            labels=['Other', 'Day', 'Evening'],
            right=False,
            include_lowest=True
        ).astype(object).fillna('Other')
    
        return meta_idx
    
        
    
    
    def filter_exec_psds(exec_spec_path, 
                         day_time=None,
                         day_time_meta_path=None,
                         stim_type=None, 
                         stim_label=None, 
                         gender=None, 
                         age=None, 
                         handiness=None):
        """
        Filters experimental PSD data (.npz file) based on subject and trial metadata.
    
        :param: exec_spec_path : str - Path to the `.npz` file containing experimental PSD data.
        :param: day_time : str - Daytime condition "Day" or "Evening"
        :param: day_time_meta_path : str - Path to an Excel file with metadata. Required only if `day_time` is provided.
        :param: stim_type : str or list(str) - "g" → geometric or "r" → random.
        :param: stim_label : int or list(int) - Label(s) of stimuli ("-1" → random or "0–12" → geometric).
        :param: gender : str - Gender "m" → male or  "f" → female.
        :param: age : int or list(int) - Age(s).
        :param: handiness : str - Hand preference "r" → right-handed or "l" → left-handed
    
        :return: list - filtered execution spectras [power, phase, subject_id, trial_id, gender, handiness, age, label, img, task_type].
        """
        
        loaded = np.load(exec_spec_path)
        results_arr = []
    
        # Load daytime metadata if filtering by daytime
        if day_time is not None:
            if day_time_meta_path is None:
                raise ValueError("You must provide 'day_time_meta_path' when 'day_time' is specified.")
            day_time_df = _get_day_time_df(day_time_meta_path)
        else:
            day_time_df = None
    
        # Normalize input filters
        if isinstance(stim_label, (list, np.ndarray)):
            stim_label = set(stim_label)
        if isinstance(age, (list, np.ndarray)):
            age = set(age)
        if isinstance(stim_type, str):
            stim_type = {stim_type}
        elif stim_type is not None:
            stim_type = set(stim_type)
    
        # Iterate through all subjects/trials in .npz file
        i = 0
        while f'power_{i}' in loaded:
            power = loaded[f'power_{i}']
            phase = loaded[f'phase_{i}']
            s_id = int(loaded[f'subject_id_{i}'])
            t_id = int(loaded[f'trial_id_{i}'])
            gend = str(loaded[f'gender_{i}'])
            hand = str(loaded[f'handiness_{i}'])
            ag = int(loaded[f'age_{i}'])
            label = int(loaded[f'label_{i}'])
            img = loaded[f'img_{i}']
            task_type = str(loaded[f'task_type_{i}'])
    
            # Apply filters
            # Filter by daytime condition
            if day_time is not None:
                cond_row = day_time_df[
                    (day_time_df["Subject_id"] == s_id) & (day_time_df["Trial_id"] == t_id)
                ]
                if cond_row.empty or cond_row["Condition"].values[0] != day_time:
                    i += 1
                    continue
    
            # Filter by stim_type
            if stim_type is not None and task_type not in stim_type:
                i += 1
                continue
    
            # Filter by stim_label
            if stim_label is not None:
                if isinstance(stim_label, set):
                    if label not in stim_label:
                        i += 1
                        continue
                elif label != stim_label:
                    i += 1
                    continue
    
            # Filter by gender
            if gender is not None and gend != gender:
                i += 1
                continue
    
            # Filter by age
            if age is not None:
                if isinstance(age, set):
                    if ag not in age:
                        i += 1
                        continue
                elif ag != age:
                    i += 1
                    continue
    
            # Filter by handiness
            if handiness is not None and hand != handiness:
                i += 1
                continue
    
            results_arr.append([power, phase, s_id, t_id, gend, hand, ag, label, img, task_type])
            i += 1
    
        return results_arr

In [3]:
def _aggregate_by_subject(results):
    # subj_band_vals[(ch, band)][sid] -> [vals_per_trial...]
    subj_band_vals = defaultdict(lambda: defaultdict(list))
    for power, phase, s_id, t_id, gender, handiness, age, label, img, task_type in results:
        s_id = int(s_id)
        vecs = (power.mean(axis=2) if power.ndim == 3 else power).astype(np.float32, copy=False)  # (C,F)
        for ch in range(n_channels):
            for band, idx in band_cols.items():
                if idx.size == 0: 
                    continue
                val_trial = float(np.nanmean(vecs[ch, idx]))
                subj_band_vals[(ch, band)][s_id].append(val_trial)
    # среднее по триалам → массивы
    subj_vecs = {}
    for ch in range(n_channels):
        for band in bands.keys():
            ids, vals = [], []
            d = subj_band_vals[(ch, band)]
            for sid, arr in d.items():
                if arr:
                    ids.append(sid)
                    vals.append(float(np.nanmean(arr)))
            subj_vecs[(ch, band)] = {'ids': ids, 'vals': np.array(vals, float)}
    return subj_vecs

def build_subject_vectors(
    day_subj: Dict[Tuple[int, str], Dict[str, Any]],
    night_subj: Dict[Tuple[int, str], Dict[str, Any]],
    bands: Dict[str, Tuple[float, float]],
    n_channels: int,
) -> Dict[Tuple[int, str], Dict[str, Any]]:
    """
    Собирает subject_vectors для каждого (канал, бэнд) без парного выравнивания.

    Parameters
    ----------
    day_subj, night_subj : dict
        {(ch, band): {'ids': List[int], 'vals': np.ndarray}, ...}
        — результат aggregate_by_subject(...)
    bands : dict
        Словарь бэндов; используются только ключи (имена бэндов).
    n_channels : int
        Количество каналов.

    Returns
    -------
    subject_vectors : dict
        {(ch, band): {
            'day': np.ndarray, 'night': np.ndarray,
            'day_ids': List[int], 'night_ids': List[int]
        }}
    """
    subject_vectors: Dict[Tuple[int, str], Dict[str, Any]] = {}

    for ch in range(n_channels):
        for band in bands.keys():
            key = (ch, band)
            d = day_subj.get(key, {'ids': [], 'vals': np.array([], dtype=float)})
            n = night_subj.get(key, {'ids': [], 'vals': np.array([], dtype=float)})

            d_ids: List[int] = list(d.get('ids', []))
            n_ids: List[int] = list(n.get('ids', []))
            d_vals = np.asarray(d.get('vals', np.array([], float)), dtype=float)
            n_vals = np.asarray(n.get('vals', np.array([], float)), dtype=float)

            subject_vectors[key] = {
                'day'      : d_vals,
                'night'    : n_vals,
                'day_ids'  : d_ids,
                'night_ids': n_ids,
            }

    return subject_vectors

In [4]:
def hedges_g(x, y) -> float:
    """
    Hedges' g для двух независимых выборок (Day − Night), с поправкой J.
    """
    x = np.asarray(x, float); y = np.asarray(y, float)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan
    vx, vy = np.var(x, ddof=1), np.var(y, ddof=1)
    # обе дисперсии нулевые
    if vx == 0 and vy == 0:
        return float(np.sign(np.nanmean(x) - np.nanmean(y)) * 0.0)
    # объединённая (пул) СКО
    sp2 = ((nx - 1) * vx + (ny - 1) * vy) / (nx + ny - 2)
    if sp2 <= 0:
        return np.nan
    d = (np.nanmean(x) - np.nanmean(y)) / np.sqrt(sp2)  # Day − Night
    J = 1 - 3 / (4 * (nx + ny) - 9) if (nx + ny) > 2 else 1.0
    return float(J * d)


def build_band_table(subject_vectors: Dict[Tuple[int, str], Dict[str, Any]],
                     alpha: float = 0.05) -> pd.DataFrame:
    """
    Строит сводную таблицу по всем (канал, бэнд) из subject_vectors:
      Welch t-test, p-value, Hedges' g, примерная мощность (TTestIndPower).
    """
    power_calc = TTestIndPower()
    rows = []

    for (ch, band), vecs in subject_vectors.items():
        x = np.asarray(vecs.get('day',   []), float)   # day
        y = np.asarray(vecs.get('night', []), float)   # night
        n1, n2 = len(x), len(y)
        if n1 == 0 or n2 == 0:
            continue

        # Welch t-test (устойчивей при разн. дисперсиях)
        try:
            t_stat, p_val = stats.ttest_ind(x, y, equal_var=False, nan_policy='omit')
        except Exception:
            t_stat, p_val = np.nan, np.nan

        # Эффект (Hedges' g), знак Day − Night
        g = hedges_g(x, y)

        # Прикидка мощности (допущение равных дисперсий в модели мощности)
        try:
            ratio = n2 / max(n1, 1)
            es = abs(g)
            power = power_calc.solve_power(effect_size=es, nobs1=n1, ratio=ratio,
                                           alpha=alpha, alternative='two-sided')
        except Exception:
            power = np.nan

        rows.append({
            'channel': ch,
            'band': band,
            'n_day': n1,
            'n_night': n2,
            'mean_day': float(np.nanmean(x)) if n1 else np.nan,
            'mean_night': float(np.nanmean(y)) if n2 else np.nan,
            'delta_day_minus_night': float(np.nanmean(x) - np.nanmean(y)) if (n1 and n2) else np.nan,
            't_stat': float(t_stat) if np.isfinite(t_stat) else np.nan,
            'p_value': float(p_val) if np.isfinite(p_val) else np.nan,
            'hedges_g': float(g) if np.isfinite(g) else np.nan,
            'power': float(power) if np.isfinite(power) else np.nan,
            'sig_alpha_0.05': bool((p_val <= 0.05) if np.isfinite(p_val) else False),
            'sig_alpha_0.01': bool((p_val <= 0.01) if np.isfinite(p_val) else False),
            'sig_and_power': bool((p_val <= 0.05) and (power >= 0.8)) if (np.isfinite(p_val) and np.isfinite(power)) else False,
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(['band', 'p_value', 'power'],
                            ascending=[True, True, False]).reset_index(drop=True)
    return df

# --- пример использования ---
# band_table = build_band_table(subject_vectors, alpha=0.05)
# print(band_table.head(12))




In [5]:
# === 1) Грузим сразу готовые поднаборы: Day и Evening (будем называть Night) ===
exec_spec_path     = r'./Generated/Spectrums/exec_and_rest_morlets.npz'
day_time_meta_path = r"./Supplementary/Experiment_Metadata.xlsx"

results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, stim_type="g")
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, stim_type="g")  # Evening ~ Night

print(f"Day trials: {len(results_day)}, Night trials: {len(results_night)}")

# Размерности из первого доступного набора
first_power = (results_day or results_night)[0][0]
n_channels, n_freqs = first_power.shape[:2]

# Частоты (если нет внешнего freqs/psd_freqs)
if 'freqs' in globals() and len(freqs) == n_freqs:
    f = np.asarray(freqs)
elif 'psd_freqs' in globals() and len(psd_freqs) == n_freqs:
    f = np.asarray(psd_freqs)
else:
    f = np.linspace(0.0, 60.0, n_freqs, endpoint=False)  # fallback

bands = {'Delta':(1,4), 'Tetta':(4,7), 'Alpha':(7,13), 'Beta':(13,30)}
band_cols = {name: np.where((f >= lo) & (f < hi))[0] for name,(lo,hi) in bands.items()}


Day trials: 420, Night trials: 420


In [6]:

# (b) Вектора по субъектам (НЕ парные): усредняем по триалам субъекта в каждом бэнде


day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

# Итоговая структура как раньше (раздельные day/night, без выравнивания пар)
subject_vectors = {}
for ch in range(n_channels):
    for band in bands.keys():
        subject_vectors[(ch, band)] = {
            'day'      : day_subj[(ch, band)]['vals'],
            'night'    : night_subj[(ch, band)]['vals'],
            'day_ids'  : day_subj[(ch, band)]['ids'],
            'night_ids': night_subj[(ch, band)]['ids'],
        }

# Пример проверки
#k = (0, 'Alpha')
#print("Alpha, ch0: day_n=", subject_vectors[k]['day'].size, " night_n=", subject_vectors[k]['night'].size)


In [7]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.power import TTestIndPower

alpha = 0.05
power_calc = TTestIndPower()

def hedges_g(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan
    vx, vy = np.var(x, ddof=1), np.var(y, ddof=1)
    if vx == 0 and vy == 0:
        return np.sign(np.mean(x)-np.mean(y)) * 0.0
    sp = np.sqrt(((nx-1)*vx + (ny-1)*vy) / (nx + ny - 2))
    if sp == 0:
        return np.nan
    d = (np.mean(x) - np.mean(y)) / sp                   # Day − Night
    J = 1 - 3/(4*(nx + ny) - 9) if (nx+ny) > 2 else 1.0  # small-sample correction
    return J * d

rows = []
for (ch, band), vecs in subject_vectors.items():
    x = np.asarray(vecs['day'], float)    # длина ≈ 9
    y = np.asarray(vecs['night'], float)  # длина ≈ 7

    n1, n2 = len(x), len(y)
    if n1 == 0 or n2 == 0:
        continue

    # Welch's t-test
    try:
        t_stat, p_val = stats.ttest_ind(x, y, equal_var=False, nan_policy='omit')
    except Exception:
        t_stat, p_val = np.nan, np.nan

    # Effect size (Hedges' g) — знак = Day − Night
    g = hedges_g(x, y)

    # Power (приближение равных дисперсий в power-анализе — допустимо как грубая оценка)
    try:
        ratio = n2 / max(n1, 1)
        power = power_calc.solve_power(effect_size=abs(g), nobs1=n1, ratio=ratio,
                                       alpha=alpha, alternative='two-sided')
    except Exception:
        power = np.nan

    rows.append({
        'channel': ch,
        'band': band,
        'n_day': n1,
        'n_night': n2,
        'mean_day': float(np.nanmean(x)) if n1 else np.nan,
        'mean_night': float(np.nanmean(y)) if n2 else np.nan,
        'delta_day_minus_night': float(np.nanmean(x) - np.nanmean(y)) if (n1 and n2) else np.nan,
        't_stat': float(t_stat) if np.isfinite(t_stat) else np.nan,
        'p_value': float(p_val) if np.isfinite(p_val) else np.nan,
        'hedges_g': float(g) if np.isfinite(g) else np.nan,
        'power': float(power) if np.isfinite(power) else np.nan,
        'sig_alpha_0.05': bool((p_val <= 0.05) if np.isfinite(p_val) else False),
        'sig_alpha_0.01': bool((p_val <= 0.01) if np.isfinite(p_val) else False),
        'sig_and_power': bool((p_val <= 0.05) and (power >= 0.8)) if (np.isfinite(p_val) and np.isfinite(power)) else False,
    })

band_table = pd.DataFrame(rows).sort_values(['band', 'p_value', 'power'], ascending=[True, True, False]).reset_index(drop=True)
print(band_table.head(12))


    channel   band  n_day  n_night   mean_day  mean_night  \
0         8  Alpha     18       16   0.141507    0.269105   
1        23  Alpha     18       16   0.136226    0.094062   
2        17  Alpha     18       16   0.271821    0.193755   
3        51  Alpha     18       16   0.210720    0.158796   
4        11  Alpha     18       16   0.138159    0.104824   
5        10  Alpha     18       16   0.131124    0.096273   
6        61  Alpha     18       16   0.154185    0.204696   
7        32  Alpha     18       16   0.146757    0.115710   
8        15  Alpha     18       16   0.267892    0.205469   
9        60  Alpha     18       16   0.148669    0.120700   
10       58  Alpha     18       16   0.140799    0.175038   
11       30  Alpha     18       16  11.107169    0.542181   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0               -0.127598 -1.523026  0.145733 -0.537569  0.329400   
1                0.042164  1.338496  0.192849  0.432102  0.230411   

In [8]:
results_day_r   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, stim_type="r")
results_night_r = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, stim_type="r")

day_subj   = _aggregate_by_subject(results_day_r)
night_subj = _aggregate_by_subject(results_night_r)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))


    channel   band  n_day  n_night  mean_day  mean_night  \
0         1  Alpha     18       16  0.256933    0.167304   
1        20  Alpha     18       16  0.190325    0.290286   
2        11  Alpha     18       16  0.184640    0.118855   
3        23  Alpha     18       16  0.134537    0.098372   
4        40  Alpha     18       16  0.135693    0.200049   
5        37  Alpha     18       16  0.144429    0.111094   
6        55  Alpha     18       16  0.126504    0.103262   
7         8  Alpha     18       16  0.140114    0.179813   
8        58  Alpha     18       16  0.147161    0.193335   
9        15  Alpha     18       16  0.297376    0.210491   
10       32  Alpha     18       16  0.149058    0.122560   
11       16  Alpha     18       16  0.217291    0.178987   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0                0.089629  1.641730  0.113209  0.530081  0.321813   
1               -0.099961 -1.576890  0.128256 -0.545339  0.337350   
2           

In [9]:
results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, stim_type="g")
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, stim_type="r")

day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))


    channel   band  n_day  n_night   mean_day  mean_night  \
0        20  Alpha     18       16   0.167163    0.290286   
1         1  Alpha     18       16   0.238365    0.167304   
2        23  Alpha     18       16   0.136226    0.098372   
3        41  Alpha     18       16   0.171209    0.123413   
4        24  Alpha     18       16   0.139830    0.183186   
5        58  Alpha     18       16   0.140799    0.193335   
6        59  Alpha     18       16   0.249334    0.478184   
7        30  Alpha     18       16  11.107169    0.403423   
8        61  Alpha     18       16   0.154185    0.200114   
9         8  Alpha     18       16   0.141507    0.179813   
10       45  Alpha     18       16   0.182006    0.227766   
11       40  Alpha     18       16   0.151841    0.200049   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0               -0.123123 -2.035009  0.055107 -0.710744  0.518517   
1                0.071060  1.293197  0.207855  0.417424  0.218147   

In [10]:
results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, stim_type="r")
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, stim_type="g")

day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))

    channel   band  n_day  n_night  mean_day  mean_night  \
0        11  Alpha     18       16  0.184640    0.104824   
1         8  Alpha     18       16  0.140114    0.269105   
2        17  Alpha     18       16  0.309565    0.193755   
3        23  Alpha     18       16  0.134537    0.094062   
4        40  Alpha     18       16  0.135693    0.197196   
5        55  Alpha     18       16  0.126504    0.097845   
6        54  Alpha     18       16  0.116692    0.090707   
7        32  Alpha     18       16  0.149058    0.115710   
8        37  Alpha     18       16  0.144429    0.109302   
9        10  Alpha     18       16  0.123600    0.096273   
10       21  Alpha     18       16  0.132499    0.098911   
11       60  Alpha     18       16  0.152924    0.120700   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0                0.079816  1.838448  0.078898  0.590670  0.385122   
1               -0.128991 -1.542096  0.141159 -0.544516  0.336505   
2           

In [11]:
results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, gender='m')
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, gender='m')

day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))

    channel   band  n_day  n_night  mean_day  mean_night  \
0        38  Alpha      5       10  0.090671    0.064642   
1        54  Alpha      5       10  0.116308    0.075424   
2        13  Alpha      5       10  0.151747    0.105077   
3        11  Alpha      5       10  0.143835    0.074603   
4        39  Alpha      5       10  0.101745    0.071786   
5        10  Alpha      5       10  0.113238    0.082513   
6        23  Alpha      5       10  0.136618    0.076024   
7        37  Alpha      5       10  0.120878    0.079351   
8        22  Alpha      5       10  0.097184    0.072113   
9        21  Alpha      5       10  0.121244    0.075084   
10       52  Alpha      5       10  0.128251    0.074507   
11        7  Alpha      5       10  0.118174    0.067411   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0                0.026029  2.278995  0.040205  0.940235  0.356034   
1                0.040884  2.225329  0.050549  1.064672  0.436580   
2           

In [12]:
results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, gender='f')
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, gender='f')

day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))

    channel   band  n_day  n_night   mean_day  mean_night  \
0        61  Alpha     13        5   0.165439    0.312619   
1         8  Alpha     13        5   0.146409    0.426977   
2        12  Alpha     13        5   0.170898    0.325840   
3        56  Alpha     13        5   0.113791    0.165376   
4        58  Alpha     13        5   0.147978    0.252773   
5        40  Alpha     13        5   0.138538    0.247791   
6        34  Alpha     13        5   0.168601    0.229041   
7         2  Alpha     13        5   0.145634    0.198948   
8        30  Alpha     13        5  10.294786    0.568753   
9        45  Alpha     13        5   0.209114    0.306721   
10       46  Alpha     13        5   0.202634    0.286674   
11        3  Alpha     13        5   0.189644    0.268929   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0               -0.147180 -2.198577  0.065293 -1.150067  0.537284   
1               -0.280567 -2.058070  0.101689 -1.507639  0.767231   

In [13]:
results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, stim_type="r", gender='m')
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, stim_type="r", gender='m')

day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))

    channel   band  n_day  n_night  mean_day  mean_night  \
0        38  Alpha      5       10  0.094482    0.067620   
1        55  Alpha      5       10  0.118042    0.081496   
2        23  Alpha      5       10  0.142603    0.078666   
3        37  Alpha      5       10  0.116509    0.081808   
4        22  Alpha      5       10  0.107133    0.076691   
5        54  Alpha      5       10  0.120360    0.081264   
6        41  Alpha      5       10  0.163681    0.090313   
7        32  Alpha      5       10  0.142642    0.082615   
8         7  Alpha      5       10  0.119532    0.071202   
9        11  Alpha      5       10  0.204008    0.079257   
10        1  Alpha      5       10  0.283743    0.121863   
11       21  Alpha      5       10  0.125742    0.081527   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0                0.026863  2.241763  0.043127  0.932819  0.351395   
1                0.036547  1.912310  0.081721  0.870686  0.313463   
2           

In [14]:
results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, stim_type="g", gender='m')
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, stim_type="g", gender='m')

day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))

    channel   band  n_day  n_night  mean_day  mean_night  \
0        38  Alpha      5       10  0.088766    0.063154   
1        54  Alpha      5       10  0.114282    0.072504   
2        39  Alpha      5       10  0.101710    0.069131   
3        13  Alpha      5       10  0.157825    0.102588   
4        10  Alpha      5       10  0.112980    0.079902   
5         2  Alpha      5       10  0.149949    0.087718   
6        43  Alpha      5       10  0.150030    0.104272   
7        52  Alpha      5       10  0.125644    0.070575   
8        21  Alpha      5       10  0.118996    0.071862   
9         7  Alpha      5       10  0.117495    0.065516   
10       23  Alpha      5       10  0.133626    0.074703   
11       37  Alpha      5       10  0.123063    0.078122   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0                0.025612  2.240921  0.043200  0.933009  0.351513   
1                0.041777  2.343446  0.043834  1.161920  0.501608   
2           

In [15]:
results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, stim_type="r", gender='m')
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, stim_type="g", gender='m')

day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))

    channel   band  n_day  n_night  mean_day  mean_night  \
0        38  Alpha      5       10  0.094482    0.063154   
1        39  Alpha      5       10  0.101814    0.069131   
2        54  Alpha      5       10  0.120360    0.072504   
3        22  Alpha      5       10  0.107133    0.069824   
4        37  Alpha      5       10  0.116509    0.078122   
5        10  Alpha      5       10  0.113752    0.079902   
6        23  Alpha      5       10  0.142603    0.074703   
7        27  Alpha      5       10  0.116213    0.081083   
8        21  Alpha      5       10  0.125742    0.071862   
9        55  Alpha      5       10  0.118042    0.079820   
10       52  Alpha      5       10  0.133464    0.070575   
11        7  Alpha      5       10  0.119532    0.065516   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0                0.031329  2.698006  0.018417  1.135767  0.484054   
1                0.032683  2.407470  0.032598  0.936417  0.353642   
2           

In [16]:
results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, gender='m')
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, gender='f')

day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))

    channel   band  n_day  n_night  mean_day  mean_night  \
0        61  Alpha      5        5  0.135545    0.312619   
1        34  Alpha      5        5  0.130037    0.229041   
2        12  Alpha      5        5  0.123975    0.325840   
3        47  Alpha      5        5  0.158882    0.350040   
4         8  Alpha      5        5  0.127090    0.426977   
5        58  Alpha      5        5  0.129769    0.252773   
6        50  Alpha      5        5  0.121651    0.248770   
7        48  Alpha      5        5  0.175015    0.328482   
8         6  Alpha      5        5  0.106952    0.172216   
9        27  Alpha      5        5  0.106121    0.170342   
10       18  Alpha      5        5  0.119794    0.235758   
11       46  Alpha      5        5  0.143826    0.286674   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0               -0.177074 -2.890084  0.035498 -1.650961  0.629753   
1               -0.099005 -2.533633  0.049133 -1.447338  0.520643   
2           

In [17]:
results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, stim_type="g", gender='m')
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, stim_type="r", gender='m')

day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))

    channel   band  n_day  n_night  mean_day  mean_night  \
0        38  Alpha      5       10  0.088766    0.067620   
1        41  Alpha      5       10  0.172165    0.090313   
2        13  Alpha      5       10  0.157825    0.110056   
3        54  Alpha      5       10  0.114282    0.081264   
4         2  Alpha      5       10  0.149949    0.099451   
5        23  Alpha      5       10  0.133626    0.078666   
6         7  Alpha      5       10  0.117495    0.071202   
7        37  Alpha      5       10  0.123063    0.081808   
8        10  Alpha      5       10  0.112980    0.087737   
9        32  Alpha      5       10  0.134325    0.082615   
10        1  Alpha      5       10  0.175817    0.121863   
11       20  Alpha      5       10  0.175712    0.272312   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0                0.021146  1.791121  0.096580  0.737561  0.239027   
1                0.081852  2.048531  0.097714  1.350510  0.625995   
2           

In [18]:
results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, gender='f')
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, gender='m')

day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))

    channel   band  n_day  n_night  mean_day  mean_night  \
0        32  Alpha     13       10  0.151534    0.079682   
1        11  Alpha     13       10  0.157429    0.074603   
2        60  Alpha     13       10  0.154879    0.088150   
3        33  Alpha     13       10  0.155602    0.093137   
4        47  Alpha     13       10  0.273935    0.149129   
5        37  Alpha     13       10  0.137824    0.079351   
6        34  Alpha     13       10  0.168601    0.100023   
7        42  Alpha     13       10  0.171310    0.098915   
8         2  Alpha     13       10  0.145634    0.091629   
9        28  Alpha     13       10  0.155791    0.099778   
10       38  Alpha     13       10  0.110217    0.064642   
11        6  Alpha     13       10  0.154748    0.090250   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0                0.071853  2.348591  0.032667  0.853749  0.490735   
1                0.082826  2.238375  0.042664  0.801546  0.443775   
2           

In [19]:
results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, stim_type="g", gender='f')
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, stim_type="g", gender='f')

day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))

    channel   band  n_day  n_night   mean_day  mean_night  \
0        61  Alpha     13        5   0.163057    0.325356   
1        56  Alpha     13        5   0.111861    0.175871   
2        12  Alpha     13        5   0.167142    0.326592   
3         8  Alpha     13        5   0.147573    0.542194   
4        58  Alpha     13        5   0.145511    0.258127   
5        34  Alpha     13        5   0.164863    0.224333   
6        40  Alpha     13        5   0.140165    0.291752   
7        14  Alpha     13        5   0.166951    0.229494   
8        30  Alpha     13        5  15.283064    0.563319   
9         2  Alpha     13        5   0.144410    0.195918   
10       45  Alpha     13        5   0.197031    0.284386   
11       46  Alpha     13        5   0.189739    0.265137   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0               -0.162298 -2.100219  0.082341 -1.212588  0.581170   
1               -0.064010 -1.767808  0.109994 -0.795470  0.295486   

In [20]:
results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, stim_type="r", gender='f')
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, stim_type="r", gender='f')

day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))

    channel   band  n_day  n_night  mean_day  mean_night  \
0        47  Alpha     13        5  0.257748    0.397785   
1        12  Alpha     13        5  0.178412    0.324335   
2        58  Alpha     13        5  0.152911    0.242066   
3        61  Alpha     13        5  0.170203    0.287146   
4         3  Alpha     13        5  0.193438    0.293874   
5        34  Alpha     13        5  0.176075    0.238457   
6         2  Alpha     13        5  0.148083    0.205006   
7        45  Alpha     13        5  0.233281    0.351390   
8        18  Alpha     13        5  0.176128    0.264706   
9        46  Alpha     13        5  0.228423    0.329748   
10        4  Alpha     13        5  0.180457    0.272595   
11       42  Alpha     13        5  0.172173    0.281404   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0               -0.140037 -1.384102  0.196990 -0.607845  0.192404   
1               -0.145923 -1.333565  0.239779 -0.861710  0.337354   
2           

In [21]:
results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, stim_type="g", gender='f')
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, stim_type="r", gender='f')

day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))

    channel   band  n_day  n_night  mean_day  mean_night  \
0        46  Alpha     13        5  0.189739    0.329748   
1        59  Alpha     13        5  0.226849    0.558214   
2        45  Alpha     13        5  0.197031    0.351390   
3        34  Alpha     13        5  0.164863    0.238457   
4        12  Alpha     13        5  0.167142    0.324335   
5        58  Alpha     13        5  0.145511    0.242066   
6        61  Alpha     13        5  0.163057    0.287146   
7         3  Alpha     13        5  0.187747    0.293874   
8        47  Alpha     13        5  0.282028    0.397785   
9         2  Alpha     13        5  0.144410    0.205006   
10       18  Alpha     13        5  0.171492    0.264706   
11       42  Alpha     13        5  0.170878    0.281404   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0               -0.140009 -1.550995  0.170672 -0.857378  0.334544   
1               -0.331364 -1.604940  0.174195 -1.124538  0.519200   
2           

In [22]:
results_day   = filter_exec_psds(exec_spec_path, day_time="Day",     day_time_meta_path=day_time_meta_path, stim_type="r", gender='f')
results_night = filter_exec_psds(exec_spec_path, day_time="Evening", day_time_meta_path=day_time_meta_path, stim_type="g", gender='f')

day_subj   = _aggregate_by_subject(results_day)
night_subj = _aggregate_by_subject(results_night)

subject_vectors = build_subject_vectors(day_subj, night_subj, bands, n_channels)

band_table_r = build_band_table(subject_vectors, alpha=0.05)
print(band_table_r.head(12))

    channel   band  n_day  n_night  mean_day  mean_night  \
0        61  Alpha     13        5  0.170203    0.325356   
1         8  Alpha     13        5  0.144082    0.542194   
2        12  Alpha     13        5  0.178412    0.326592   
3        56  Alpha     13        5  0.117651    0.175871   
4        58  Alpha     13        5  0.152911    0.258127   
5        40  Alpha     13        5  0.135282    0.291752   
6        41  Alpha     13        5  0.135371    0.247161   
7        34  Alpha     13        5  0.176075    0.224333   
8         2  Alpha     13        5  0.148083    0.195918   
9        59  Alpha     13        5  0.401690    0.261910   
10       30  Alpha     13        5  0.318229    0.563319   
11       36  Alpha     13        5  0.152108    0.212724   

    delta_day_minus_night    t_stat   p_value  hedges_g     power  \
0               -0.155153 -1.973425  0.094820 -1.092539  0.496479   
1               -0.398112 -1.902728  0.126825 -1.514618  0.771006   
2           